## Dataset Selection

For this project, the **PaySim Synthetic Financial Dataset** (available on Kaggle: `ealaxi/paysim1`) was selected as the primary data source for fraud detection.

PaySim is a synthetic dataset simulating mobile money transactions, generated using real transaction data from a private dataset provided by a multinational company. It was designed specifically to enable fraud detection research while preserving the statistical properties of real-world financial data.

### Reasons for Selection

**1. Relevance to the Research Paper**
This dataset was one of the four datasets used in Chung & Lee (2023), which proposed an improved fraud detection strategy using KNN, LDA, and Linear Regression. Using the same dataset allows for direct comparison and validation of results against the published findings.

**2. Scale and Richness**
The dataset contains approximately **6 million transactions**, providing sufficient volume for training and evaluating machine learning models with statistical confidence.

**3. Labeled Fraud Transactions**
Each transaction is labeled with a binary fraud indicator (`isFraud`), making it directly suitable for supervised classification tasks without additional annotation.

**4. Feature Diversity**
The dataset includes multiple transaction types (CASH-IN, CASH-OUT, DEBIT, PAYMENT, TRANSFER) along with features such as transaction amount, account balances before and after the transaction, and sender/receiver identifiers — providing a rich feature set for model training.

**5. Class Imbalance Reflects Reality**
The dataset is heavily imbalanced, with fraudulent transactions accounting for approximately 0.13% of all records. This mirrors real-world fraud patterns and presents a meaningful challenge for evaluating model recall — the primary metric emphasised in Chung & Lee (2023).

### Reference
Chung, J., & Lee, K. (2023). Credit Card Fraud Detection: An Improved Strategy for High Recall Using KNN, LDA, and Linear Regression. *Sensors (Basel, Switzerland)*, 23(18), 7788. https://doi.org/10.3390/s23187788

In [ ]:
import os
import json

# Generate Json
kaggle_dir = os.path.expanduser("~/.config/kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_config = {
    "username": "username",  
    "key": "KGAT_xxxxx"           
    }

with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
    json.dump(kaggle_config, f)

os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
print("Json generated!")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ealaxi/paysim1")
print("Path to dataset files:", path)

In [ ]:
import pandas as pd

# check dataset
df = pd.read_csv(path + "/PS_20174392719_1491204439457_log.csv")
print(df.shape)
print(df.head())
print(df['isFraud'].value_counts())


In [ ]:
# record fraud rate
fraud_rate = df['isFraud'].mean()
print(f"Fraud rate: {fraud_rate:.4%}")

In [ ]:
# print columns name
print(df.columns.tolist())

In [ ]:
# Prepressing data type
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

# Drop any columns that are pure IDs or fully null
df.dropna(axis=1, how='all', inplace=True)

# Check basic stat
print(df['type'].value_counts())
print(df.isnull().sum())
print(df.dtypes)
print(df.columns)

In [ ]:
# class imbalance check
fraud_rate = df['isFraud'].mean() * 100
print(f'Fraud rate: {fraud_rate:.2f}%')

# If fraud_rate < 20%, you have class imbalance.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

# isFlaggedFraud 是系统标记，不是真实标签，删掉避免数据泄露
df.drop(columns=['isFlaggedFraud'], inplace=True)

print(df.head())

In [ ]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

print(f"Features shape: {X.shape}")
print(f"Fraud cases: {y.sum()} / {len(y)} ({y.mean():.4%})")

In [ ]:
from sklearn.model_selection import train_test_split

# 第1步：删列
df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], 
        inplace=True, errors='ignore')
print(df.columns)

# 第2步：重新定义 X 和 y ← 这步你漏掉了！
X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 第3步：分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean():.4%}")
print(f"Test fraud rate: {y_test.mean():.4%}")

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"After SMOTE - Train size: {X_train_res.shape}")
print(f"After SMOTE - Fraud rate: {y_train_res.mean():.4%}")

In [ ]:
# training nn for bank fraud detection
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)  # 注意：用训练集的scaler来转换测试集

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(16, activation='relu'),
    
    Dense(1, activation='sigmoid')  # 二分类输出
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['recall', 'accuracy']  # 重点看 recall
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_recall',
    patience=3,
    mode='max',             # recall 越高越好
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train_res,
    epochs=50,
    batch_size=2048,        # 数据量大，用大batch
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test_scaled) > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Feature Statistics
print(df.describe())

variances = df.var(numeric_only=True).sort_values(ascending=False)
print("\nTop variances:")
print(variances.head(10))

In [ ]:
#Correlation with Fraud
correlations = df.corr(numeric_only=True)['isFraud'].abs().sort_values(ascending=False)
print("Top correlations with isFraud:")
print(correlations.head(10))

## EDA Summary

- **Dataset shape:** 6,362,620 rows × 9 columns
- **Fraud rate:** 0.13% — severely imbalanced
- **Missing values:** None
- **Preprocessing done:**
  - Encoded `type` column using LabelEncoder
  - Dropped `isFlaggedFraud` (system flag, not a real label)
  - Applied SMOTE (sampling_strategy=0.3) on training set only
  - Standardized features using StandardScaler
- **Most useful features (by correlation with isFraud):**
  - `amount`, `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`

## Methodology

This project uses the PaySim Synthetic Financial Dataset, which simulates 
mobile money transactions and contains approximately 6.36 million records 
with binary fraud labels. The dataset is severely imbalanced, with a fraud 
rate of only 0.13%, which reflects real-world fraud patterns.

Preprocessing involved removing non-predictive identifier columns, encoding 
the categorical transaction type feature using LabelEncoder, and applying 
StandardScaler to normalise all features to zero mean and unit variance. 
To address class imbalance, SMOTE (Synthetic Minority Oversampling Technique) 
was applied exclusively to the training set with a sampling strategy of 0.3, 
bringing the fraud rate to approximately 23% while keeping the test set 
untouched to ensure realistic evaluation.

The model architecture is a fully connected Neural Network (NN) built with 
TensorFlow/Keras, consisting of three hidden layers (64, 32, and 16 neurons) 
with ReLU activations, Batch Normalisation, and Dropout (0.3) for 
regularisation. This architecture was chosen for its ability to capture 
non-linear relationships between features — particularly important given that 
all individual feature correlations with fraud were below 0.08.

Given the severe class imbalance, accuracy alone is a misleading metric. 
The primary evaluation metric is Recall, as missing a fraudulent transaction 
carries a higher cost than a false alarm. Precision, F1 Score, and AUC-ROC 
are also reported to provide a comprehensive view of model performance. 
This aligns with the findings of Chung & Lee (2023), who demonstrated that 
optimising for recall is critical in credit card fraud detection tasks.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

# Classification report
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

# AUC-ROC
auc = roc_auc_score(y_test, y_pred_prob)
print(f"AUC-ROC Score: {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Recall
axes[1].plot(history.history['recall'], label='Train Recall')
axes[1].plot(history.history['val_recall'], label='Val Recall')
axes[1].set_title('Recall over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Recall')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='tomato', label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()

In [ ]:
model.save_weights('paysim_client_weights.weights.h5')
print("Weights saved!")

In [ ]:
model.save('paysim_model.keras')
print("Model saved!")

## Model Results

### Performance Metrics (Fraud Class)
| Metric    | Score  |
|-----------|--------|
| Accuracy  | 1.00   |
| Precision | 0.21   |
| Recall    | 0.95   |
| F1 Score  | 0.34   |
| AUC-ROC   | 0.9989 |

### Confusion Matrix Breakdown
- **True Positives (Real frauds caught):** 1,556 out of 1,643
- **False Negatives (Frauds missed):** 87
- **False Positives (Legitimate flagged wrongly):** 5,909
- **True Negatives (Legitimate correctly identified):** 1,264,972

### Key Takeaway
The model successfully detected 95% of all fraudulent transactions, 
missing only 87 out of 1,643 real fraud cases. The low Precision (0.21) 
indicates a higher false alarm rate, which is an acceptable trade-off 
when the priority is to minimise missed fraud — consistent with the 
high-recall strategy proposed by Chung & Lee (2023).

In [ ]:
import joblib
import numpy as np

# 保存 scaler
joblib.dump(scaler, 'scaler.pkl')

# 保存处理好的测试集
np.save('X_test_scaled.npy', X_test_scaled)
np.save('y_test.npy', y_test)

# 保存 SMOTE 后的训练集
np.save('X_train_res.npy', X_train_res)
np.save('y_train_res.npy', y_train_res)

print("All files saved!")

In [ ]:
import os

# 查看当前文件夹里所有文件
files = os.listdir('.')
for f in files:
    if f.endswith(('.h5', '.keras', '.pkl', '.npy')):
        print(f)

In [ ]:
# 在 paysim_train.ipynb 里运行
import numpy as np
from sklearn.preprocessing import StandardScaler

# 保存原始训练集（SMOTE之前的）
X_train_scaled_original = scaler.transform(X_train)  # 用已有的scaler
np.save('X_train_original.npy', X_train_scaled_original)
np.save('y_train_original.npy', y_train.values)
print("Saved!")